# 🎬 HyperFrames + Whisper en Google Colab

Este notebook hace todo automáticamente:
1. Monta **Google Drive** (tu almacenamiento).
2. Instala dependencias (faster-whisper, moviepy, etc.).
3. Detecta palabras habladas en tu video (ej. "casa", "storm").
4. Genera `motions.json` con los tiempos exactos.
5. Renderiza los gráficos sobre el video.
6. Guarda el resultado **de vuelta en Drive**.

---
### 📁 Estructura en Drive:
```
MiDrive/
└── hyperframes/
    ├── input/     ← sube aquí tu video (input.mp4)
    ├── output/    ← aquí queda resultado.mp4
    └── config/    ← motions.json generado
```


In [ ]:
# ============================================
# CELDA 1: Montar Drive e instalar dependencias
# ============================================

from google.colab import drive
import os, sys, subprocess

drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/hyperframes"
INPUT_DIR = os.path.join(DRIVE_ROOT, "input")
OUTPUT_DIR = os.path.join(DRIVE_ROOT, "output")
CONFIG_DIR = os.path.join(DRIVE_ROOT, "config")
SCRIPTS_DIR = "/content/hyperframes/scripts"

for d in [INPUT_DIR, OUTPUT_DIR, CONFIG_DIR, SCRIPTS_DIR]:
    os.makedirs(d, exist_ok=True)

!pip install -q faster-whisper "moviepy==1.0.3" Pillow imageio-ffmpeg

print("✅ Drive montado y dependencias instaladas")
print(f"📂 Input:  {INPUT_DIR}")
print(f"📂 Output: {OUTPUT_DIR}")
print(f"📂 Config: {CONFIG_DIR}")


---
### 📤 Antes de continuar:
Sube tu video a Drive en la carpeta `MiDrive/hyperframes/input/` con el nombre `input.mp4`.

In [ ]:
# ============================================
# CELDA 2: Crear add_motion_graphics.py (motor de render)
# ============================================

add_motion_code = r'''"""add_motion_graphics.py para Colab"""
import argparse, json, os, sys, gc, math
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from moviepy.editor import VideoFileClip, VideoClip

def hex_to_rgba(h, a=1.0):
    h = h.lstrip("#")
    if len(h) == 6:
        return tuple(int(h[i:i+2], 16) for i in (0,2,4)) + (int(255*a),)
    return (255,255,255,int(255*a))

def get_font(size, bold=False):
    paths = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    ]
    for p in paths:
        if os.path.exists(p):
            return ImageFont.truetype(p, size)
    return ImageFont.load_default()

def parse_pos(pos, W, H, default_y=None):
    if default_y is None:
        default_y = H // 2
    if not isinstance(pos, (list, tuple)) or len(pos) < 2:
        return (W // 2, default_y)
    x = W // 2 if pos[0] == "center" else (int(pos[0]) if isinstance(pos[0], (int, float)) else W // 2)
    if pos[1] == "center":
        y = H // 2
    elif pos[1] == "bottom":
        y = H - 160
    elif pos[1] == "top":
        y = 160
    elif isinstance(pos[1], (int, float)):
        y = int(pos[1])
    else:
        y = default_y
    return (x, y)

def effect_lower_third(W, H, fx):
    start, end = fx["start"], fx["end"]
    dur = end - start
    color = hex_to_rgba(fx.get("color", "#FFFFFF"))
    bg = hex_to_rgba(fx.get("bg_color", "#000000"), fx.get("bg_opacity", 0.75))
    fontsize = fx.get("fontsize", 42)
    offset_y = fx.get("offset_y", 140)
    pos_y = H - offset_y
    text = fx.get("text", "")
    font = get_font(fontsize, bold=True)
    dummy = Image.new("RGBA", (10, 10))
    draw = ImageDraw.Draw(dummy)
    bbox = draw.textbbox((0, 0), text, font=font, stroke_width=1)
    pad = 24
    tw = (bbox[2] - bbox[0]) + 2 * pad
    th = (bbox[3] - bbox[1]) + 2 * pad
    def make_frame(t):
        img = Image.new("RGBA", (W, H), (0, 0, 0, 0))
        draw = ImageDraw.Draw(img)
        bx = (W - tw) // 2
        by = pos_y - th // 2
        draw.rectangle([bx, by, bx + tw, by + th], fill=bg)
        draw.text((bx + pad - bbox[0], by + pad - bbox[1]), text, font=font, fill=color, stroke_width=1)
        return np.array(img)
    clip = VideoClip(make_frame, duration=dur)
    return clip.set_start(start)

def effect_highlight(W, H, fx):
    start, end = fx["start"], fx["end"]
    dur = end - start
    center = parse_pos(fx.get("position", ["center", "center"]), W, H, default_y=H // 2)
    radius = fx.get("radius", 110)
    color = hex_to_rgba(fx.get("color", "#FFD700"))
    opacity = fx.get("opacity", 0.45)
    anim = fx.get("animation", "pulse")
    def make_frame(t):
        img = Image.new("RGBA", (W, H), (0, 0, 0, 0))
        draw = ImageDraw.Draw(img)
        r = radius
        if anim == "pulse":
            r = radius * (1.0 + 0.18 * math.sin(2 * math.pi * t * 3))
        bbox = [center[0] - r, center[1] - r, center[0] + r, center[1] + r]
        fill = (*color[:3], int(color[3] * opacity))
        draw.ellipse(bbox, fill=fill, outline=(*color[:3], 230), width=4)
        return np.array(img)
    clip = VideoClip(make_frame, duration=dur)
    return clip.set_start(start)

def effect_arrow(W, H, fx):
    start, end = fx["start"], fx["end"]
    dur = end - start
    p1 = parse_pos(fx.get("from", [100, H - 200]), W, H)
    p2 = parse_pos(fx.get("to", [W - 100, H - 200]), W, H)
    color = hex_to_rgba(fx.get("color", "#FF0000"))
    width = fx.get("width", 10)
    anim = fx.get("animation", "draw")
    def make_frame(t):
        img = Image.new("RGBA", (W, H), (0, 0, 0, 0))
        draw = ImageDraw.Draw(img)
        prog = min(t / dur, 1.0) if anim == "draw" else 1.0
        x2 = p1[0] + (p2[0] - p1[0]) * prog
        y2 = p1[1] + (p2[1] - p1[1]) * prog
        draw.line([p1[0], p1[1], x2, y2], fill=color, width=width)
        if prog > 0.05:
            angle = math.atan2(p2[1] - p1[1], p2[0] - p1[0])
            tip = width * 3
            pL = (x2 - tip * math.cos(angle - math.pi / 6), y2 - tip * math.sin(angle - math.pi / 6))
            pR = (x2 - tip * math.cos(angle + math.pi / 6), y2 - tip * math.sin(angle + math.pi / 6))
            draw.polygon([(x2, y2), pL, pR], fill=color)
        return np.array(img)
    clip = VideoClip(make_frame, duration=dur)
    return clip.set_start(start)

def effect_comparison(W, H, fx):
    start, end = fx["start"], fx["end"]
    dur = end - start
    split_x = fx.get("split_x", 0.5)
    left_label = fx.get("left_label", "ANTES")
    right_label = fx.get("right_label", "DESPUÉS")
    label_color = hex_to_rgba(fx.get("label_color", "#FFFFFF"))
    def make_frame(t):
        img = Image.new("RGBA", (W, H), (0, 0, 0, 0))
        draw = ImageDraw.Draw(img)
        x = int(W * split_x)
        draw.rectangle([x, 0, W, H], fill=(0, 0, 0, 110))
        draw.line([x, 0, x, H], fill=(255, 255, 255, 200), width=4)
        font = get_font(48, bold=True)
        draw.text((40, 40), left_label, font=font, fill=label_color)
        draw.text((x + 40, 40), right_label, font=font, fill=label_color)
        return np.array(img)
    clip = VideoClip(make_frame, duration=dur)
    return clip.set_start(start)

def effect_callout(W, H, fx):
    start, end = fx["start"], fx["end"]
    dur = end - start
    text = fx.get("text", "")
    bx, by = parse_pos(fx.get("position", [100, 300]), W, H)
    arrow_to = parse_pos(fx.get("arrow_to", [W // 2, H // 2]), W, H)
    bg_color = hex_to_rgba(fx.get("bg_color", "#0066FF"))
    fontsize = fx.get("fontsize", 36)
    font = get_font(fontsize, bold=True)
    dummy = Image.new("RGBA", (10, 10))
    draw = ImageDraw.Draw(dummy)
    bbox = draw.textbbox((0, 0), text, font=font, stroke_width=1)
    pad = 18
    tw = (bbox[2] - bbox[0]) + 2 * pad
    th = (bbox[3] - bbox[1]) + 2 * pad
    def make_frame(t):
        img = Image.new("RGBA", (W, H), (0, 0, 0, 0))
        draw = ImageDraw.Draw(img)
        draw.line([bx + tw // 2, by + th, arrow_to[0], arrow_to[1]], fill=(255, 255, 255, 220), width=4)
        draw.rectangle([bx, by, bx + tw, by + th], fill=bg_color, outline=(255, 255, 255, 220), width=2)
        draw.text((bx + pad - bbox[0], by + pad - bbox[1]), text, font=font, fill=(255, 255, 255, 255))
        return np.array(img)
    clip = VideoClip(make_frame, duration=dur)
    return clip.set_start(start)

def build_clips(W, H, effects):
    clips = []
    for fx in effects:
        ft = fx.get("type")
        if ft == "lower_third": clips.append(effect_lower_third(W, H, fx))
        elif ft == "highlight": clips.append(effect_highlight(W, H, fx))
        elif ft == "arrow": clips.append(effect_arrow(W, H, fx))
        elif ft == "comparison": clips.append(effect_comparison(W, H, fx))
        elif ft == "callout": clips.append(effect_callout(W, H, fx))
    return clips

def ensure_rgba(frame):
    if frame.ndim == 3 and frame.shape[2] == 4:
        return frame
    if frame.ndim == 3 and frame.shape[2] == 3:
        alpha = np.full((frame.shape[0], frame.shape[1], 1), 255, dtype=np.uint8)
        return np.dstack([frame, alpha])
    return frame

def render_video(input_path, output_path, effects, fps=24):
    base = VideoFileClip(input_path)
    W, H = base.size
    overlay_clips = build_clips(W, H, effects)
    def compose_frame(t):
        frame = base.get_frame(t)
        frame = ensure_rgba(frame).astype(np.float32) / 255.0
        for clip in overlay_clips:
            if t < clip.start or t > clip.end:
                continue
            rel_t = t - clip.start
            if rel_t < 0 or rel_t > clip.duration:
                continue
            ov = clip.get_frame(rel_t)
            ov = ensure_rgba(ov)
            if ov.shape[:2] != (H, W):
                ov = np.array(Image.fromarray(ov.astype(np.uint8)).resize((W, H), Image.LANCZOS))
                ov = ensure_rgba(ov)
            ov = ov.astype(np.float32) / 255.0
            alpha = ov[:, :, 3:4]
            rgb = ov[:, :, :3]
            frame[:, :, :3] = frame[:, :, :3] * (1.0 - alpha) + rgb * alpha
        return (frame[:, :, :3] * 255).astype(np.uint8)
    final_clip = VideoClip(compose_frame, duration=base.duration)
    final_clip = final_clip.set_fps(fps)
    if base.audio is not None:
        final_clip = final_clip.set_audio(base.audio)
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
    final_clip.write_videofile(output_path, fps=fps, codec="libx264",
                               audio_codec="aac", temp_audiofile="temp-audio.m4a",
                               remove_temp=True, verbose=False, logger=None)
    print(f"✅ Video renderizado: {output_path}")
'''

with open(os.path.join(SCRIPTS_DIR, "add_motion_graphics.py"), "w", encoding="utf-8") as f:
    f.write(add_motion_code)

print("✅ add_motion_graphics.py creado")


In [ ]:
# ============================================
# CELDA 3: Pipeline detectar + render
# ============================================

import gc, json, tempfile, wave, subprocess
import numpy as np
import imageio_ffmpeg
from faster_whisper import WhisperModel

FFMPEG_EXE = imageio_ffmpeg.get_ffmpeg_exe()

def extract_audio(video_path, wav_path):
    cmd = [FFMPEG_EXE, "-y", "-i", video_path,
           "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1", wav_path]
    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def load_wav_as_array(wav_path):
    with wave.open(wav_path, "rb") as wf:
        n_channels = wf.getnchannels()
        sample_width = wf.getsampwidth()
        n_frames = wf.getnframes()
        raw = wf.readframes(n_frames)
    dtype = np.int16 if sample_width == 2 else np.int32
    audio = np.frombuffer(raw, dtype=dtype)
    if n_channels > 1:
        audio = audio.reshape(-1, n_channels)[:, 0]
    return audio.astype(np.float32) / 32768.0

def transcribe_words(audio_path, model_size="tiny", language=None):
    # En Colab CPU para tiny tarda < 1s y es 100% inmune a errores de driver CUDA
    device = "cpu"
    compute_type = "int8"
    if model_size in ["medium", "large", "large-v3"]:
        try:
            import torch
            if torch.cuda.is_available():
                torch.zeros(1).cuda()
                device = "cuda"
                compute_type = "float16"
        except Exception:
            device = "cpu"
            compute_type = "int8"
    print(f"🎤 Transcribiendo con {model_size} en {device.upper()} ({compute_type})...")
    model = WhisperModel(model_size, device=device, compute_type=compute_type)
    audio_array = load_wav_as_array(audio_path)
    segments, info = model.transcribe(
        audio_array, word_timestamps=True, language=language,
        vad_filter=True, beam_size=1,
    )
    words = []
    for segment in segments:
        for w in segment.words:
            words.append({"word": w.word, "start": round(float(w.start), 3), "end": round(float(w.end), 3)})
    del model
    gc.collect()
    return words, info

def build_motions(words, keywords, effect_type="highlight", duration=1.5):
    effects = []
    for w in words:
        palabra = w["word"].strip().lower()
        if any(k.lower() in palabra for k in keywords):
            effects.append({
                "type": effect_type,
                "start": w["start"],
                "end": round(w["start"] + duration, 3),
                "text": w["word"],
                "position": ["center", "center"],
                "offset_y": 140,
                "fontsize": 42,
                "color": "#FFFFFF",
                "bg_color": "#000000",
                "bg_opacity": 0.85,
                "animation": "pulse",
            })
    return effects

def pipeline_hyperframes(video_path, keyword, model_size="tiny", language="es",
                         effect="highlight", duration=1.5, output_path=None):
    if output_path is None:
        output_path = os.path.join(OUTPUT_DIR, "resultado.mp4")
    tmp_fd, wav_path = tempfile.mkstemp(suffix=".wav")
    os.close(tmp_fd)
    try:
        print("🎵 Extrayendo audio...")
        extract_audio(video_path, wav_path)
        words, info = transcribe_words(wav_path, model_size, language)
        print(f"✅ Idioma detectado: {info.language} | Palabras: {len(words)}")
        print(f"🔍 Buscando: '{keyword}'")
        effects = build_motions(words, [keyword], effect, duration)
        print(f"✅ Coincidencias encontradas: {len(effects)}")
        if not effects:
            print("⚠️ No se encontró la palabra en el audio.")
            return None
        config = {"input": video_path, "output": output_path, "fps": 24, "effects": effects}
        os.makedirs(CONFIG_DIR, exist_ok=True)
        with open(os.path.join(CONFIG_DIR, "motions.json"), "w", encoding="utf-8") as f:
            json.dump(config, f, indent=2, ensure_ascii=False)
        print("💾 Config guardado")
        print("🎬 Renderizando video...")
        sys.path.insert(0, SCRIPTS_DIR)
        from add_motion_graphics import render_video
        render_video(video_path, output_path, effects, fps=24)
        return output_path
    finally:
        if os.path.exists(wav_path):
            os.remove(wav_path)

print("✅ pipeline_hyperframes listo")


In [ ]:
# ============================================
# CELDA 4: Ejecutar (cambia aquí tus parámetros)
# ============================================

VIDEO_FILENAME = "input.mp4"
KEYWORD = "casa"           # palabra a detectar
LANGUAGE = "es"            # es / en / None
MODEL = "tiny"             # tiny o base
EFFECT = "highlight"       # highlight / lower_third / arrow / comparison / callout
DURATION = 1.5

video_path = os.path.join(INPUT_DIR, VIDEO_FILENAME)
output_path = os.path.join(OUTPUT_DIR, "resultado.mp4")

if not os.path.exists(video_path):
    print(f"❌ No se encontró: {video_path}")
    print("Sube tu video a Drive: MiDrive/hyperframes/input/input.mp4")
else:
    resultado = pipeline_hyperframes(
        video_path=video_path, keyword=KEYWORD,
        model_size=MODEL, language=LANGUAGE,
        effect=EFFECT, duration=DURATION,
        output_path=output_path,
    )
    if resultado:
        print(f"\n🎉 LISTO. Video en Drive: {resultado}")
        from IPython.display import FileLink
        display(FileLink(resultado))
